# BIOT 6900 · Module 3 Lab, Part 1: A Toy Protein Language Model

**In-class lab · Week 3 · runs offline in seconds · no downloads**

Real protein language models have hundreds of millions of parameters, so you cannot see inside them. Tonight you build a **toy** version small enough to understand completely, and use it to walk through every idea from the lecture:

| Section | Lecture idea |
|---|---|
| 1 · A toy protein world | Proteins as a language; families; conservation |
| 2 · Hide a residue, predict it | The masked-prediction training task |
| 3 · Embeddings and clusters | What an embedding is; similar biology clusters |
| 4 · A toy druggability prior | Labels + classifier; homology leakage; labels encode history |
| 5 · Variant scoring | log P(mutant) − log P(wild-type) |

**Everything here is synthetic.** The families, sequences, and "druggability" labels are invented so that you know the right answer and can check the method against it. Next session you run the same steps on real human proteins.

**How to work:** run each cell in order (Shift+Enter). Cells marked **✎ YOUR TURN** ask you to change one value or write one sentence. Uses only `numpy` and `pandas` (already in your `biot6900` environment); plots appear if `matplotlib` is installed and are skipped otherwise.

In [1]:
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except ImportError:
    HAVE_PLT = False
    print("matplotlib not found: plots will be skipped, tables still print.")

rng = np.random.default_rng(6900)
AA = list("ACDEFGHIKLMNPQRSTVWY")
print("Ready. numpy", np.__version__, "| pandas", pd.__version__, "| plots:", HAVE_PLT)

matplotlib not found: plots will be skipped, tables still print.
Ready. numpy 2.4.6 | pandas 3.0.6 | plots: False


## 1 · A toy protein world

We invent **twelve protein families**, each with 15 members of length 40. Each family has a consensus sequence. Members copy the consensus but mutate freely, **except at 10 conserved positions** that never change (think: catalytic or structural residues).

Six families are "druggable". Five of them share a **pocket motif** `GSGKST` at positions 15–20 (think of a pocket shape that chemists have learned to target). The sixth druggable family, *Channel-like*, is druggable through a **different** pocket that no other family has. Remember that; it matters in Section 4.

In [2]:
L, N_PER = 40, 15
FAMILIES = {                 # name: druggable?
    "Kinase-like": True, "GPCR-like": True, "Protease-like": True, "NucReceptor-like": True,
    "Transporter-like": True, "Channel-like": True,
    "TF-like": False, "Structural-like": False, "Ribosomal-like": False,
    "Scaffold-like": False, "RNAbinding-like": False, "Chaperone-like": False,
}
SHARED_POCKET = ["Kinase-like", "GPCR-like", "Protease-like", "NucReceptor-like", "Transporter-like"]
POCKET = "GSGKST"            # shared by five druggable families at positions 15-20
ALT_POCKET = "WYHFPW"        # Channel-like uses a different pocket

consensus, conserved, rows = {}, {}, []
for fam, drug in FAMILIES.items():
    c = list(rng.choice(AA, L))
    if fam in SHARED_POCKET:
        c[14:20] = list(POCKET)
    if fam == "Channel-like":
        c[14:20] = list(ALT_POCKET)
    consensus[fam] = "".join(c)
    cons_pos = set(rng.choice([p for p in range(L) if not 14 <= p < 20], 10, replace=False)) | set(range(14, 20))
    conserved[fam] = cons_pos
    for k in range(N_PER):
        s = [a if (p in cons_pos or rng.random() > 0.5) else rng.choice(AA) for p, a in enumerate(c)]
        rows.append({"name": f"{fam.split('-')[0]}_{k+1:02d}", "family": fam, "druggable": drug, "seq": "".join(s)})

world = pd.DataFrame(rows)
print(world.groupby("family")["druggable"].agg(["size", "first"]).rename(columns={"size": "members", "first": "druggable"}))
print("\nFirst five Kinase-like members (pocket motif at positions 15-20):")
for s in world[world.family == "Kinase-like"].seq.head(5):
    print("  ", s[:14], s[14:20], s[20:])

                  members  druggable
family                              
Channel-like           15       True
Chaperone-like         15      False
GPCR-like              15       True
Kinase-like            15       True
NucReceptor-like       15       True
Protease-like          15       True
RNAbinding-like        15      False
Ribosomal-like         15      False
Scaffold-like          15      False
Structural-like        15      False
TF-like                15      False
Transporter-like       15       True

First five Kinase-like members (pocket motif at positions 15-20):
   ECSVKLEHIAFHAL GSGKST WWGNWPHHINVPEVWEISVR
   LVSVWHRLLAFHSL GSGKST WYGNMRLGTNVFKTSRNCVR
   GYSVKAWQVAFWFL GSGKST WYGNWGHCIDVHKVWAMWVE
   GVSVLLDPMAFWYL GSGKST WQGNNPHHIDVLKVCEIGVS
   GWSVKDGFVAFWSL GSGKST WYGNNPHANWVYYVMEICVR


**✎ YOUR TURN (1 sentence):** Look at the five sequences above. Which positions are identical in all of them, and why would evolution keep those positions fixed?

## 2 · Hide a residue, predict it

This is the training task from the lecture. Our toy "language model" predicts a hidden residue in two steps:

1. **Read the context:** compare the rest of the sequence with each family's consensus to work out which family it belongs to.
2. **Predict:** use how often each amino acid appears at that position in that family.

A real model learns this from data with no family labels, but the logic is the same: **the context tells you what kind of protein this is, and that tells you what belongs in the gap.**

In [3]:
# Column frequencies per family (with a small pseudocount so nothing has zero probability)
FREQ = {}
for fam in FAMILIES:
    seqs = world[world.family == fam].seq
    counts = np.full((L, 20), 0.5)
    for s in seqs:
        for p, a in enumerate(s):
            counts[p, AA.index(a)] += 1
    FREQ[fam] = counts / counts.sum(1, keepdims=True)

def predict_hidden(seq, pos, beta=0.5):
    # Return P(amino acid) at `pos`, using only the rest of the sequence as context.
    fams = list(FAMILIES)
    ident = np.array([sum(a == b for p, (a, b) in enumerate(zip(seq, consensus[f])) if p != pos) for f in fams])
    w = np.exp(beta * (ident - ident.max())); w /= w.sum()          # how strongly the context points to each family
    probs = sum(wi * FREQ[f][pos] for wi, f in zip(w, fams))
    return pd.Series(probs, index=AA).sort_values(ascending=False), pd.Series(w, index=fams).round(3)

def show_prediction(seq, pos):
    probs, fam_w = predict_hidden(seq, pos)
    masked = seq[:pos] + "?" + seq[pos + 1:]
    print("Sequence :", masked)
    print(f"Hidden   : position {pos + 1}, true residue = {seq[pos]}")
    print("Context says family:", fam_w.idxmax(), f"(weight {fam_w.max():.2f})")
    print("Top 5 predictions  :", ", ".join(f"{a} {p:.2f}" for a, p in probs.head(5).items()))
    if HAVE_PLT:
        probs.head(8)[::-1].plot.barh(figsize=(5, 2.6), color="#1C7293", title=f"P(residue) at position {pos + 1}")
        plt.xlabel("probability"); plt.tight_layout(); plt.show()

query = world[world.family == "Kinase-like"].seq.iloc[0]
cpos = sorted(conserved["Kinase-like"] - set(range(14, 20)))[0]     # a conserved position
show_prediction(query, cpos)

Sequence : EC?VKLEHIAFHALGSGKSTWWGNWPHHINVPEVWEISVR
Hidden   : position 3, true residue = S
Context says family: Kinase-like (weight 1.00)
Top 5 predictions  : S 0.62, W 0.02, G 0.02, H 0.02, L 0.02


**✎ YOUR TURN:** Now hide a **variable** position instead. Run the cell below, then answer: *Why is the model confident at a conserved position but unsure at a variable one?* (This is exactly the "conservation" signal from lecture slide 6.)

In [4]:
vpos = [p for p in range(L) if p not in conserved["Kinase-like"]][0]   # ✎ try other variable positions too
show_prediction(query, vpos)

Sequence : ?CSVKLEHIAFHALGSGKSTWWGNWPHHINVPEVWEISVR
Hidden   : position 1, true residue = E
Context says family: Kinase-like (weight 1.00)
Top 5 predictions  : G 0.38, E 0.10, Y 0.06, M 0.06, P 0.06


**What the toy cannot do:** real models also learn **co-variation**, where two distant positions change together because they touch in the folded protein (lecture slide 7). Our toy treats every position independently, so it can't. That is one reason real models need hundreds of millions of parameters.

## 3 · Embeddings and clusters

An embedding is a list of numbers that places each protein at a location, so similar proteins land near each other (lecture slide 8). Our toy embedding: count every pair of neighbouring amino acids (400 possible pairs), then compress to **64 numbers** per protein. Real ESM C embeddings have 960.

We then squash the 16 numbers down to 2 so we can look at them (PCA). **The families were never given to the embedding**; watch whether they cluster anyway.

In [5]:
PAIRS = {a + b: i for i, (a, b) in enumerate((a, b) for a in AA for b in AA)}
P16 = np.random.default_rng(1).normal(size=(400, 64))

def embed(seq):
    v = np.zeros(400)
    for k in range(len(seq) - 1):
        v[PAIRS[seq[k:k + 2]]] += 1
    return v @ P16

X = np.stack([embed(s) for s in world.seq])
print("Embedding matrix:", X.shape, "(proteins x numbers)")
print("Protein 1, first 10 of 64 numbers:", np.round(X[0][:10], 1))

Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
xy = Xc @ Vt[:2].T
world["pc1"], world["pc2"] = xy[:, 0], xy[:, 1]

# Nearest neighbour check: is each protein's closest protein from the same family?
Z = X / np.linalg.norm(X, axis=1, keepdims=True)
Ssim = Z @ Z.T; np.fill_diagonal(Ssim, -np.inf)
nn_same = (world.family.values[Ssim.argmax(1)] == world.family.values).mean()
print(f"\nNearest neighbour is in the same family for {nn_same:.0%} of proteins (chance = {1/len(FAMILIES):.0%})")

if HAVE_PLT:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for fam, g in world.groupby("family"):
        ax.scatter(g.pc1, g.pc2, s=22, label=fam, marker="o" if FAMILIES[fam] else "^")
    ax.set_title("Toy embeddings, 2D projection (circles = druggable)"); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=7, ncol=2, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

Embedding matrix: (180, 64) (proteins x numbers)
Protein 1, first 10 of 64 numbers: [-4.1  2.1 -7.8 -3.2 -0.7  6.7 -2.3  3.2 -1.2 -6.4]

Nearest neighbour is in the same family for 94% of proteins (chance = 8%)


**✎ YOUR TURN (1 sentence):** The embedding was never told the family names. Why do the families still form clusters?

## 4 · A toy druggability prior

Now the lecture's four steps (slide 12): **labels → embed → train → apply.** We train a logistic regression on the embeddings to predict "druggable", and test it two ways:

* **Random split:** hold out random proteins. Their close relatives are still in training.
* **Family split:** hold out a **whole family** at a time, so the model must judge a family it has never seen.

The gap between the two is the **homology leakage** lesson (slide 14).

In [6]:
def train_logreg(X, y, lam=1e-2, steps=800, lr=0.1):
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Xs = (X - mu) / sd
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(Xs @ w + b)))
        w -= lr * (Xs.T @ (p - y) / len(y) + lam * w)
        b -= lr * (p - y).mean()
    return lambda Xn: 1 / (1 + np.exp(-(((Xn - mu) / sd) @ w + b)))

y = world.druggable.values.astype(float)

# Random split: 5 folds
folds = rng.permutation(len(world)) % 5
pred_random = np.zeros(len(world))
for f in range(5):
    tr, te = folds != f, folds == f
    pred_random[te] = train_logreg(X[tr], y[tr])(X[te])

# Family split: leave one family out
pred_family = np.zeros(len(world))
for fam in FAMILIES:
    te = (world.family == fam).values
    pred_family[te] = train_logreg(X[~te], y[~te])(X[te])

acc = lambda p: ((p > 0.5) == y).mean()
print(f"Accuracy, random split : {acc(pred_random):.0%}")
print(f"Accuracy, family split : {acc(pred_family):.0%}")

world["p_random"], world["p_family"] = pred_random, pred_family
summary = world.groupby("family").agg(druggable=("druggable", "first"),
                                      p_random=("p_random", "mean"), p_family=("p_family", "mean")).round(2)
print("\nMean predicted probability of 'druggable', per family:")
print(summary)

if HAVE_PLT:
    ax = summary[["p_random", "p_family"]].plot.bar(figsize=(7, 3.2), color=["#1C7293", "#C8553D"], rot=20)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--"); ax.set_ylabel("mean P(druggable)")
    ax.set_title("Random split vs family split"); plt.tight_layout(); plt.show()

Accuracy, random split : 96%
Accuracy, family split : 78%

Mean predicted probability of 'druggable', per family:
                  druggable  p_random  p_family
family                                         
Channel-like           True      0.88      0.40
Chaperone-like        False      0.05      0.32
GPCR-like              True      0.99      0.98
Kinase-like            True      0.93      0.77
NucReceptor-like       True      0.96      0.89
Protease-like          True      0.90      0.72
RNAbinding-like       False      0.11      0.31
Ribosomal-like        False      0.17      0.40
Scaffold-like         False      0.05      0.08
Structural-like       False      0.13      0.55
TF-like               False      0.03      0.08
Transporter-like       True      0.90      0.84


**✎ YOUR TURN (answer in 2–3 sentences each):**

1. Why is the random-split accuracy so much higher than the family-split accuracy? Which one should you report, and why?
2. Look at **Channel-like** in the family split. It *is* druggable, but the model scores it low when it has never seen it. Why? (Hint: which pocket does it use?) Connect this to the lecture point that **labels encode history** and the model under-rates "the next undruggable protein".
3. *Bonus:* one non-druggable family scores above 0.5 in the family split. Which one, and what does a false positive like this cost a real discovery team?

## 5 · Variant scoring: how surprised is the model?

The variant score from lecture slide 22:

`score = log P(mutant) − log P(wild-type)`

Hide the position, ask the model for probabilities, compare the mutant with the wild type. **Strongly negative = the model is surprised = probably poorly tolerated.** No labels and no training beyond what the model already learned.

In [7]:
def variant_score(seq, pos, mut):
    probs, _ = predict_hidden(seq, pos)
    wt = seq[pos]
    return float(np.log(probs[mut]) - np.log(probs[wt])), wt

tests = [("conserved position", cpos), ("pocket position", 16), ("variable position", vpos)]
out = []
for label, pos in tests:
    wt = query[pos]
    mut = "W" if wt != "W" else "A"
    s, _ = variant_score(query, pos, mut)
    out.append({"where": label, "variant": f"{wt}{pos + 1}{mut}", "score": round(s, 2)})
print(pd.DataFrame(out).to_string(index=False))

             where variant  score
conserved position     S3W  -3.43
   pocket position    G17W  -3.43
 variable position     E1W  -1.61


**✎ YOUR TURN:** Before running the next cell, **predict** whether each score will be strongly negative or near zero. Then change `pos` and `mut` to test your own variant.

In [8]:
pos, mut = 5, "P"          # ✎ position is 0-based here: 5 means residue 6
s, wt = variant_score(query, pos, mut)
print(f"{wt}{pos + 1}{mut}: score = {s:.2f}", "(conserved)" if pos in conserved["Kinase-like"] else "(variable)")

L6P: score = -1.30 (variable)


## Wrap-up: three questions to answer before you leave

Write one or two sentences for each in a markdown cell below.

1. **The core idea:** In your own words, how does predicting hidden residues teach a model about conservation and protein families?
2. **The limit:** Our toy scored *Channel-like* as undruggable even though it is. Name one real protein from the lecture where the same thing happened historically, and explain the parallel.
3. **The caveat:** A variant with a strongly negative score is "disfavoured by evolution". Why is that **not** the same as "clinically pathogenic"?

**Next session:** Part 2 runs these same steps on real human proteins with ESM C embeddings and real Open Targets labels, including placing p53 on the map.

*Your answers:*